In [ ]:
# ============================================================
# BLUE EAGLE SECTOR ROTATION DASHBOARD
# Momentum + Analyst Recommendation Momentum
# Google Colab | One-cell version
# UPDATED: CRSP history + yfinance extension for missing months
# ============================================================

# -----------------------------
# 0) Install packages
# -----------------------------
import sys, subprocess, pkgutil

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

required = ["psycopg2-binary", "sqlalchemy", "fredapi", "plotly", "ipywidgets", "yfinance"]
for pkg in required:
    mod = pkg.replace("-", "_")
    if pkgutil.find_loader(mod) is None:
        pip_install(pkg)

# -----------------------------
# 1) Imports
# -----------------------------
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import yfinance as yf
from sqlalchemy import create_engine
from fredapi import Fred
from getpass import getpass
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from urllib.parse import quote_plus

# -----------------------------
# 2) Parameters
# -----------------------------
TODAY = pd.Timestamp.today().normalize()
CURRENT_MONTH_END = TODAY + pd.offsets.MonthEnd(0)

START_DATE = "2018-01-01"
END_DATE   = CURRENT_MONTH_END.strftime("%Y-%m-%d")

DEFAULT_LOOKBACK_MONTHS = 6
DEFAULT_SKIP_MONTHS     = 1
DEFAULT_TOP_N           = 3
DEFAULT_BOTTOM_N        = 3

MOMENTUM_WEIGHT = 0.70
ANALYST_WEIGHT  = 0.30

SAVE_CSV_OUTPUTS = False

SECTOR_ETF_MAP = {
    "Energy": "XLE",
    "Materials": "XLB",
    "Industrials": "XLI",
    "Consumer Discretionary": "XLY",
    "Consumer Staples": "XLP",
    "Health Care": "XLV",
    "Financials": "XLF",
    "Information Technology": "XLK",
    "Communication Services": "XLC",
    "Utilities": "XLU",
    "Real Estate": "XLRE"
}

# -----------------------------
# 3) Helper functions
# -----------------------------
GICS_MAP = {
    "10": "Energy",
    "15": "Materials",
    "20": "Industrials",
    "25": "Consumer Discretionary",
    "30": "Consumer Staples",
    "35": "Health Care",
    "40": "Financials",
    "45": "Information Technology",
    "50": "Communication Services",
    "55": "Utilities",
    "60": "Real Estate"
}

def coerce_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def annualized_return(r):
    r = pd.Series(r).dropna()
    if len(r) == 0:
        return np.nan
    cum = (1 + r).prod()
    years = len(r) / 12.0
    if years <= 0 or cum <= 0:
        return np.nan
    return cum ** (1 / years) - 1

def annualized_vol(r):
    r = pd.Series(r).dropna()
    if len(r) < 2:
        return np.nan
    return r.std() * np.sqrt(12)

def sharpe_ratio(r, rf=0.0):
    ar = annualized_return(r)
    av = annualized_vol(r)
    if pd.isna(ar) or pd.isna(av) or av == 0:
        return np.nan
    return (ar - rf) / av

def max_drawdown(r):
    r = pd.Series(r).dropna()
    if len(r) == 0:
        return np.nan
    wealth = (1 + r).cumprod()
    peak = wealth.cummax()
    dd = wealth / peak - 1
    return dd.min()

def compute_compound_return(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    return np.prod(1 + x) - 1

def traffic_light_from_rank(rank, n_sectors, top_n=3, bottom_n=3):
    if pd.isna(rank):
        return np.nan
    if rank <= top_n:
        return "Green"
    elif rank > n_sectors - bottom_n:
        return "Red"
    else:
        return "Yellow"

def zscore_by_date(df, value_col, new_col):
    out = df.copy()
    def _z(x):
        sd = x.std(ddof=0)
        if pd.isna(sd) or sd == 0:
            return pd.Series(np.nan, index=x.index)
        return (x - x.mean()) / sd
    out[new_col] = out.groupby("date")[value_col].transform(_z)
    return out

def safe_hit_rate(strategy, benchmark):
    s = pd.Series(strategy)
    b = pd.Series(benchmark)
    mask = s.notna() & b.notna()
    if mask.sum() == 0:
        return np.nan
    return (s[mask] > b[mask]).mean()

def pull_yfinance_sector_returns(start_date, end_date, sector_etf_map):
    tickers = list(sector_etf_map.values())

    px_data = yf.download(
        tickers=tickers,
        start=start_date,
        end=(pd.to_datetime(end_date) + pd.Timedelta(days=7)).strftime("%Y-%m-%d"),
        auto_adjust=True,
        progress=False,
        group_by="ticker",
        threads=True
    )

    rows = []

    for sector, ticker in sector_etf_map.items():
        try:
            if ticker not in px_data:
                continue

            df_t = px_data[ticker].copy()

            if "Close" not in df_t.columns:
                continue

            s = df_t["Close"].dropna().copy()
            if len(s) == 0:
                continue

            monthly = s.resample("M").last().pct_change().dropna()

            tmp = pd.DataFrame({
                "date": monthly.index,
                "sector": sector,
                "sector_ret": monthly.values,
                "n_stocks": np.nan,
                "sector_lag_cap": np.nan,
                "data_source": "yfinance_sector_etf"
            })
            rows.append(tmp)
        except Exception as e:
            print(f"yfinance pull failed for {ticker}: {e}")

    if len(rows) == 0:
        return pd.DataFrame(columns=["date", "sector", "sector_ret", "n_stocks", "sector_lag_cap", "data_source"])

    out = pd.concat(rows, ignore_index=True)
    out["date"] = pd.to_datetime(out["date"]) + pd.offsets.MonthEnd(0)
    return out.sort_values(["sector", "date"]).reset_index(drop=True)

# -----------------------------
# 4) WRDS connection helpers
# -----------------------------
print("Enter your WRDS credentials.")
wrds_username = input("WRDS username: ").strip()
wrds_password = getpass("WRDS password: ").strip()

WRDS_HOST = "wrds-pgdata.wharton.upenn.edu"
WRDS_PORT = 9737
WRDS_DB   = "wrds"

engine = None

def connect_wrds_engine(max_tries=3, wait_seconds=5):
    global engine
    for attempt in range(1, max_tries + 1):
        try:
            password_escaped = quote_plus(wrds_password)
            conn_str = (
                f"postgresql+psycopg2://{wrds_username}:{password_escaped}"
                f"@{WRDS_HOST}:{WRDS_PORT}/{WRDS_DB}"
            )
            engine = create_engine(
                conn_str,
                connect_args={
                    "sslmode": "require",
                    "connect_timeout": 30,
                    "application_name": "colab_sector_rotation_multifactor",
                    "keepalives": 1,
                    "keepalives_idle": 30,
                    "keepalives_interval": 10,
                    "keepalives_count": 5,
                },
                pool_pre_ping=True,
                pool_recycle=180,
            )
            with engine.connect() as conn:
                conn.exec_driver_sql("select 1")
            print("WRDS connection successful.")
            return engine
        except Exception as e:
            print(f"WRDS connection failed on attempt {attempt}: {e}")
            if attempt < max_tries:
                time.sleep(wait_seconds)
            else:
                raise

def run_wrds_sql(sql, date_cols=None, max_tries=3, wait_seconds=5):
    global engine
    for attempt in range(1, max_tries + 1):
        try:
            return pd.read_sql_query(sql, engine, parse_dates=date_cols)
        except Exception as e:
            print(f"Query failed on attempt {attempt}: {e}")
            if attempt < max_tries:
                print("Reconnecting to WRDS...")
                try:
                    engine.dispose()
                except:
                    pass
                time.sleep(wait_seconds)
                connect_wrds_engine()
            else:
                raise

connect_wrds_engine()

# -----------------------------
# 5) Pull CRSP monthly stock data
# -----------------------------
print("Pulling CRSP monthly data...")

crsp_sql = f"""
select
    a.permno,
    a.date,
    a.ret,
    a.prc,
    a.shrout,
    b.shrcd,
    b.exchcd
from crsp.msf as a
join crsp.msenames as b
    on a.permno = b.permno
   and b.namedt <= a.date
   and a.date <= b.nameendt
where a.date between '{START_DATE}' and '{END_DATE}'
  and b.shrcd in (10, 11)
  and b.exchcd in (1, 2, 3)
"""

crsp = run_wrds_sql(crsp_sql, date_cols=["date"])
crsp = coerce_numeric(crsp, ["ret", "prc", "shrout", "shrcd", "exchcd"])
crsp = crsp.dropna(subset=["date", "permno", "ret", "prc", "shrout"]).copy()
crsp["mkt_cap"] = crsp["prc"].abs() * crsp["shrout"]
crsp["date"] = pd.to_datetime(crsp["date"]) + pd.offsets.MonthEnd(0)

print(f"CRSP rows: {len(crsp):,}")
print(f"CRSP max date: {crsp['date'].max()}")

# -----------------------------
# 6) Pull CCM link table
# -----------------------------
print("Pulling CRSP-Compustat link table...")

ccm_sql = """
select
    gvkey,
    lpermno as permno,
    linktype,
    linkprim,
    linkdt,
    linkenddt
from crsp.ccmxpf_linktable
where lpermno is not null
  and linktype in ('LU','LC','LS')
  and linkprim in ('P','C')
"""
ccm = run_wrds_sql(ccm_sql, date_cols=["linkdt", "linkenddt"])
ccm["linkenddt"] = ccm["linkenddt"].fillna(pd.Timestamp("2100-12-31"))

# -----------------------------
# 7) Pull Compustat GICS sector
# -----------------------------
print("Pulling Compustat GICS sectors...")

comp_sql = """
select
    gvkey,
    conm,
    gsector
from comp.company
where gsector is not null
"""
comp = run_wrds_sql(comp_sql)
comp["gsector"] = comp["gsector"].astype(str).str.strip()

# -----------------------------
# 8) Build stock-level base panel
# -----------------------------
print("Building stock-level base panel...")

stocks = crsp.merge(ccm, how="left", on="permno")
stocks = stocks[
    (stocks["date"] >= stocks["linkdt"]) &
    (stocks["date"] <= stocks["linkenddt"])
].copy()

stocks = stocks.sort_values(["permno", "date", "gvkey"]).drop_duplicates(["permno", "date"], keep="first")
stocks = stocks.merge(comp[["gvkey", "gsector"]], how="left", on="gvkey")

stocks["gsector"] = stocks["gsector"].astype(str).str.strip()
stocks["sector"] = stocks["gsector"].map(GICS_MAP)
stocks = stocks[stocks["sector"].notna()].copy()

stocks = stocks.sort_values(["permno", "date"]).copy()
stocks["lag_mkt_cap"] = stocks.groupby("permno")["mkt_cap"].shift(1)
stocks = stocks.dropna(subset=["lag_mkt_cap", "ret"]).copy()
stocks = stocks[stocks["lag_mkt_cap"] > 0].copy()

print(f"Stock panel rows: {len(stocks):,}")

# -----------------------------
# 9) Build sector returns from CRSP
# -----------------------------
print("Building value-weighted sector returns from CRSP...")

sector_totals = stocks.groupby(["date", "sector"], as_index=False)["lag_mkt_cap"].sum()
sector_totals = sector_totals.rename(columns={"lag_mkt_cap": "sector_lag_cap"})

stocks = stocks.merge(sector_totals, on=["date", "sector"], how="left")
stocks["weight"] = stocks["lag_mkt_cap"] / stocks["sector_lag_cap"]

sector_returns_crsp = (
    stocks.groupby(["date", "sector"], as_index=False)
    .apply(lambda x: pd.Series({
        "sector_ret": np.sum(x["weight"] * x["ret"]),
        "n_stocks": x["permno"].nunique(),
        "sector_lag_cap": x["sector_lag_cap"].iloc[0],
        "data_source": "crsp_stock_agg"
    }))
    .reset_index(drop=True)
)

print(f"CRSP sector returns max date: {sector_returns_crsp['date'].max()}")

# -----------------------------
# 10) Extend missing months with yfinance
# -----------------------------
print("Pulling missing sector returns from yfinance...")

crsp_max_date = pd.to_datetime(sector_returns_crsp["date"].max()) if len(sector_returns_crsp) > 0 else pd.to_datetime(START_DATE)
yf_start_date = (crsp_max_date - pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")

sector_returns_yf = pull_yfinance_sector_returns(
    start_date=yf_start_date,
    end_date=END_DATE,
    sector_etf_map=SECTOR_ETF_MAP
)

if len(sector_returns_yf) > 0:
    sector_returns_yf = sector_returns_yf[sector_returns_yf["date"] > crsp_max_date].copy()

print(f"yfinance sector returns rows: {len(sector_returns_yf):,}")
if len(sector_returns_yf) > 0:
    print(f"yfinance sector returns max date: {sector_returns_yf['date'].max()}")

sector_returns = pd.concat(
    [sector_returns_crsp, sector_returns_yf],
    ignore_index=True
).sort_values(["sector", "date"])

sector_returns = sector_returns.drop_duplicates(["date", "sector"], keep="first").copy()

all_dates = pd.date_range(sector_returns["date"].min(), sector_returns["date"].max(), freq="M")
all_sectors = list(GICS_MAP.values())
grid = pd.MultiIndex.from_product([all_dates, all_sectors], names=["date", "sector"]).to_frame(index=False)

sector_returns = grid.merge(sector_returns, how="left", on=["date", "sector"]).sort_values(["sector", "date"])

print(f"Combined sector return rows: {len(sector_returns):,}")
print(f"Combined sector returns max date: {sector_returns['date'].max()}")

# -----------------------------
# 11) Discover IBES recommendation table
# -----------------------------
print("Discovering IBES recommendation table...")

table_candidates = [
    ("ibes", "recdsum"),
    ("tr_ibes", "recdsum"),
    ("ibes", "recdsumus"),
    ("tr_ibes", "recdsumus"),
]

ticker_col_candidates = ["ticker", "ibtic", "oftic"]
date_col_candidates   = ["statpers", "date", "anndats"]
rec_col_candidates    = ["meanrec", "meanrecom", "mean_rec"]
nanalyst_candidates   = ["numest", "numrec", "num_analysts"]

def get_table_columns(schema_name, table_name):
    sql = f"""
    select column_name
    from information_schema.columns
    where table_schema = '{schema_name}'
      and table_name = '{table_name}'
    """
    df = run_wrds_sql(sql)
    return [c.lower() for c in df["column_name"].tolist()] if len(df) > 0 else []

def first_match(candidates, available):
    for c in candidates:
        if c in available:
            return c
    return None

rec_table_schema = None
rec_table_name   = None
rec_ticker_col   = None
rec_date_col     = None
rec_value_col    = None
rec_numest_col   = None

for schema_name, table_name in table_candidates:
    cols = get_table_columns(schema_name, table_name)
    if len(cols) == 0:
        continue
    tcol = first_match(ticker_col_candidates, cols)
    dcol = first_match(date_col_candidates, cols)
    rcol = first_match(rec_col_candidates, cols)
    ncol = first_match(nanalyst_candidates, cols)
    if tcol and dcol and rcol:
        rec_table_schema = schema_name
        rec_table_name   = table_name
        rec_ticker_col   = tcol
        rec_date_col     = dcol
        rec_value_col    = rcol
        rec_numest_col   = ncol
        break

if rec_table_schema is None:
    raise ValueError(
        "Could not find an IBES recommendation summary table automatically. "
        "Run a WRDS table search for recommendation summary data and update table_candidates."
    )

print(f"Using recommendation table: {rec_table_schema}.{rec_table_name}")
print(f"Ticker col: {rec_ticker_col} | Date col: {rec_date_col} | Rec col: {rec_value_col} | Num analysts col: {rec_numest_col}")

# -----------------------------
# 12) Pull recommendation data
# -----------------------------
print("Pulling analyst recommendation data...")

rec_start = (pd.to_datetime(START_DATE) - pd.DateOffset(months=18)).strftime("%Y-%m-%d")
extra_cols = f", {rec_numest_col}" if rec_numest_col is not None else ""

rec_sql = f"""
select
    {rec_ticker_col} as ibes_ticker,
    {rec_date_col}   as stat_date,
    {rec_value_col}  as mean_rec
    {extra_cols}
from {rec_table_schema}.{rec_table_name}
where {rec_date_col} between '{rec_start}' and '{END_DATE}'
"""

rec = run_wrds_sql(rec_sql, date_cols=["stat_date"])
rec["ibes_ticker"] = rec["ibes_ticker"].astype(str).str.strip()
rec = coerce_numeric(rec, ["mean_rec"] + ([rec_numest_col] if rec_numest_col is not None else []))
rec = rec.dropna(subset=["ibes_ticker", "stat_date", "mean_rec"]).copy()

if rec_numest_col is not None:
    rec = rec[rec[rec_numest_col].fillna(0) >= 1].copy()

rec["date"] = pd.to_datetime(rec["stat_date"]) + pd.offsets.MonthEnd(0)
rec = rec.sort_values(["ibes_ticker", "stat_date"]).drop_duplicates(["ibes_ticker", "date"], keep="last")

print(f"Recommendation rows after monthly snapshot logic: {len(rec):,}")
print(f"Recommendation max date: {rec['date'].max()}")

# -----------------------------
# 13) Discover CRSP-IBES linking table
# -----------------------------
print("Discovering IBES-CRSP link table...")

link_candidates = [
    ("wrdsapps", "ibcrsphist"),
    ("ibes", "iclink"),
    ("tr_ibes", "iclink"),
]

link_ticker_candidates = ["ticker", "ibtic"]
link_permno_candidates = ["permno", "lpermno"]
link_start_candidates  = ["sdate", "fdate", "namedt", "linkdt"]
link_end_candidates    = ["edate", "ldate", "nameendt", "linkenddt"]
link_score_candidates  = ["score"]

link_schema = None
link_table  = None
link_ticker_col = None
link_permno_col = None
link_start_col  = None
link_end_col    = None
link_score_col  = None

for schema_name, table_name in link_candidates:
    cols = get_table_columns(schema_name, table_name)
    if len(cols) == 0:
        continue
    tcol = first_match(link_ticker_candidates, cols)
    pcol = first_match(link_permno_candidates, cols)
    scol = first_match(link_start_candidates, cols)
    ecol = first_match(link_end_candidates, cols)
    qcol = first_match(link_score_candidates, cols)
    if tcol and pcol and scol and ecol:
        link_schema = schema_name
        link_table  = table_name
        link_ticker_col = tcol
        link_permno_col = pcol
        link_start_col  = scol
        link_end_col    = ecol
        link_score_col  = qcol
        break

if link_schema is None:
    raise ValueError(
        "Could not find an IBES-CRSP linking table automatically. "
        "Update link_candidates for your WRDS account."
    )

print(f"Using link table: {link_schema}.{link_table}")

extra_score = f", {link_score_col}" if link_score_col is not None else ""

link_sql = f"""
select
    {link_ticker_col} as ibes_ticker,
    {link_permno_col} as permno,
    {link_start_col}  as link_start,
    {link_end_col}    as link_end
    {extra_score}
from {link_schema}.{link_table}
where {link_ticker_col} is not null
  and {link_permno_col} is not null
"""

link = run_wrds_sql(link_sql, date_cols=["link_start", "link_end"])
link["link_end"] = link["link_end"].fillna(pd.Timestamp("2100-12-31"))
link["ibes_ticker"] = link["ibes_ticker"].astype(str).str.strip()

# -----------------------------
# 14) Map recommendations to permno
# -----------------------------
print("Mapping recommendations to CRSP permnos...")

rec_linked = rec.merge(link, how="left", on="ibes_ticker")
rec_linked = rec_linked[
    (rec_linked["date"] >= rec_linked["link_start"]) &
    (rec_linked["date"] <= rec_linked["link_end"])
].copy()

if link_score_col is not None and link_score_col in rec_linked.columns:
    rec_linked = rec_linked.sort_values(["ibes_ticker", "date", link_score_col]).drop_duplicates(
        ["ibes_ticker", "date"], keep="first"
    )
else:
    rec_linked = rec_linked.sort_values(["ibes_ticker", "date", "permno"]).drop_duplicates(
        ["ibes_ticker", "date"], keep="first"
    )

rec_linked = rec_linked.dropna(subset=["permno"]).copy()
rec_linked["permno"] = pd.to_numeric(rec_linked["permno"], errors="coerce")
rec_linked = rec_linked.dropna(subset=["permno"]).copy()
rec_linked["permno"] = rec_linked["permno"].astype(int)

print(f"Linked recommendation rows: {len(rec_linked):,}")
print(f"Linked recommendation max date: {rec_linked['date'].max()}")

# -----------------------------
# 15) Build analyst recommendation momentum factor
# -----------------------------
print("Building analyst recommendation momentum factor...")

rec_linked = rec_linked.sort_values(["permno", "date"]).copy()
rec_linked["mean_rec_lag_12m"] = rec_linked.groupby("permno")["mean_rec"].shift(12)
rec_linked["analyst_change_12m"] = rec_linked["mean_rec_lag_12m"] - rec_linked["mean_rec"]

analyst_stock = rec_linked[["permno", "date", "mean_rec", "mean_rec_lag_12m", "analyst_change_12m"]].copy()

stocks_plus = stocks.merge(
    analyst_stock,
    how="left",
    on=["permno", "date"]
)

analyst_sector = (
    stocks_plus.dropna(subset=["analyst_change_12m"])
    .groupby(["date", "sector"], as_index=False)
    .apply(lambda x: pd.Series({
        "analyst_raw": np.sum(x["lag_mkt_cap"] * x["analyst_change_12m"]) / np.sum(x["lag_mkt_cap"]),
        "analyst_names": x["permno"].nunique()
    }))
    .reset_index(drop=True)
)

sector_factors = sector_returns.merge(analyst_sector, how="left", on=["date", "sector"]).sort_values(["sector", "date"])

print(f"Sector factors max date: {sector_factors['date'].max()}")
print(f"END_DATE used in queries: {END_DATE}")

# -----------------------------
# 16) Signal engine
# -----------------------------
def build_signals(sector_df, lookback_months=6, skip_months=1, top_n=3, bottom_n=3):
    df = sector_df.copy().sort_values(["sector", "date"])

    df["momentum_raw"] = (
        df.groupby("sector")["sector_ret"]
        .transform(lambda x: x.shift(skip_months).rolling(lookback_months).apply(compute_compound_return, raw=False))
    )

    df = zscore_by_date(df, "momentum_raw", "momentum_z")
    df = zscore_by_date(df, "analyst_raw",  "analyst_z")

    df["momentum_z_filled"] = df["momentum_z"].fillna(0)
    df["analyst_z_filled"]  = df["analyst_z"].fillna(0)

    df["final_score"] = (
        MOMENTUM_WEIGHT * df["momentum_z_filled"] +
        ANALYST_WEIGHT  * df["analyst_z_filled"]
    )

    has_signal = df["momentum_raw"].notna() | df["analyst_raw"].notna()
    df["rank"] = np.nan
    df.loc[has_signal, "rank"] = df.loc[has_signal].groupby("date")["final_score"].rank(ascending=False, method="first")

    df["n_sectors"] = df.groupby("date")["sector"].transform("count")

    df["signal"] = df.apply(
        lambda r: traffic_light_from_rank(
            r["rank"],
            int(r["n_sectors"]) if not pd.isna(r["n_sectors"]) else 11,
            top_n,
            bottom_n
        ),
        axis=1
    )

    df["fwd_1m_ret"] = df.groupby("sector")["sector_ret"].shift(-1)
    return df

# -----------------------------
# 17) Backtest engine
# -----------------------------
def run_backtest(signal_df, top_n=3):
    df = signal_df.copy()
    valid = df.dropna(subset=["rank", "fwd_1m_ret"]).copy()

    valid["selected"] = valid["rank"] <= top_n

    port = valid[valid["selected"]].groupby("date", as_index=False).agg(
        strategy_ret=("fwd_1m_ret", "mean")
    )

    bench = valid.groupby("date", as_index=False).agg(
        benchmark_ret=("fwd_1m_ret", "mean")
    )

    green = valid[valid["signal"] == "Green"].groupby("date", as_index=False).agg(
        green_ret=("fwd_1m_ret", "mean")
    )

    red = valid[valid["signal"] == "Red"].groupby("date", as_index=False).agg(
        red_ret=("fwd_1m_ret", "mean")
    )

    out = (
        port.merge(bench, on="date", how="outer")
            .merge(green, on="date", how="left")
            .merge(red, on="date", how="left")
            .sort_values("date")
            .reset_index(drop=True)
    )

    out["strategy_cum"] = (1 + out["strategy_ret"].fillna(0)).cumprod()
    out["benchmark_cum"] = (1 + out["benchmark_ret"].fillna(0)).cumprod()
    out["spread_ret"] = out["green_ret"] - out["red_ret"]
    out["spread_cum"] = (1 + out["spread_ret"].fillna(0)).cumprod()

    return out

def summarize_performance(bt):
    return pd.DataFrame({
        "Metric": [
            "Annualized Return",
            "Annualized Volatility",
            "Sharpe Ratio",
            "Max Drawdown",
            "Hit Rate vs Benchmark"
        ],
        "Strategy": [
            annualized_return(bt["strategy_ret"]),
            annualized_vol(bt["strategy_ret"]),
            sharpe_ratio(bt["strategy_ret"]),
            max_drawdown(bt["strategy_ret"]),
            safe_hit_rate(bt["strategy_ret"], bt["benchmark_ret"])
        ],
        "Benchmark": [
            annualized_return(bt["benchmark_ret"]),
            annualized_vol(bt["benchmark_ret"]),
            sharpe_ratio(bt["benchmark_ret"]),
            max_drawdown(bt["benchmark_ret"]),
            np.nan
        ],
        "Green-Red Spread": [
            annualized_return(bt["spread_ret"]),
            annualized_vol(bt["spread_ret"]),
            sharpe_ratio(bt["spread_ret"]),
            max_drawdown(bt["spread_ret"]),
            np.nan
        ]
    })

def display_perf_table(perf):
    perf_fmt = perf.copy()
    pct_metrics = {"Annualized Return", "Annualized Volatility", "Max Drawdown", "Hit Rate vs Benchmark"}
    for col in ["Strategy", "Benchmark", "Green-Red Spread"]:
        perf_fmt[col] = perf_fmt.apply(
            lambda row: f"{row[col]:.2%}" if (row["Metric"] in pct_metrics and pd.notna(row[col]))
            else (f"{row[col]:.2f}" if pd.notna(row[col]) else ""),
            axis=1
        )
    display(perf_fmt)

# -----------------------------
# 18) Initial build
# -----------------------------
signals = build_signals(
    sector_factors,
    lookback_months=DEFAULT_LOOKBACK_MONTHS,
    skip_months=DEFAULT_SKIP_MONTHS,
    top_n=DEFAULT_TOP_N,
    bottom_n=DEFAULT_BOTTOM_N
)

backtest = run_backtest(signals, top_n=DEFAULT_TOP_N)

# -----------------------------
# 19) Dashboard renderer
# -----------------------------
def render_dashboard(start_date, end_date, lookback_months, skip_months, top_n, bottom_n):
    clear_output(wait=True)

    sig = build_signals(
        sector_factors,
        lookback_months=int(lookback_months),
        skip_months=int(skip_months),
        top_n=int(top_n),
        bottom_n=int(bottom_n)
    )

    bt = run_backtest(sig, top_n=int(top_n))

    loaded_min_date = pd.to_datetime(sector_factors["date"].min()) + pd.offsets.MonthEnd(0)
    loaded_max_date = pd.to_datetime(sector_factors["date"].max()) + pd.offsets.MonthEnd(0)

    start_date = pd.to_datetime(start_date) + pd.offsets.MonthEnd(0)
    end_date = pd.to_datetime(end_date) + pd.offsets.MonthEnd(0)

    adj_start_date = max(start_date, loaded_min_date)
    adj_end_date = min(end_date, loaded_max_date)

    if start_date > loaded_max_date or end_date < loaded_min_date:
        print("=" * 88)
        print("NO DATA AVAILABLE FOR THE SELECTED DATE RANGE")
        print("=" * 88)
        print(f"You selected: {start_date.date()} to {end_date.date()}")
        print(f"Loaded model data only covers: {loaded_min_date.date()} to {loaded_max_date.date()}")
        return

    sig_f = sig[(sig["date"] >= adj_start_date) & (sig["date"] <= adj_end_date)].copy()
    bt_f = bt[(bt["date"] >= adj_start_date) & (bt["date"] <= adj_end_date)].copy()

    if len(sig_f) == 0:
        print("=" * 88)
        print("NO SIGNAL DATA IN SELECTED RANGE")
        print("=" * 88)
        print(f"Requested range: {start_date.date()} to {end_date.date()}")
        print(f"Adjusted range used: {adj_start_date.date()} to {adj_end_date.date()}")
        print(f"Loaded model data range: {loaded_min_date.date()} to {loaded_max_date.date()}")
        return

    valid_current_dates = (
        sig_f.dropna(subset=["rank"])
             .groupby("date")["sector"]
             .count()
    )

    if len(valid_current_dates) == 0:
        print("=" * 88)
        print("NO VALID SIGNAL MONTH AVAILABLE IN SELECTED RANGE")
        print("=" * 88)
        print(f"Requested range: {start_date.date()} to {end_date.date()}")
        print(f"Adjusted range used: {adj_start_date.date()} to {adj_end_date.date()}")
        print(f"Loaded model data range: {loaded_min_date.date()} to {loaded_max_date.date()}")
        return

    current_date = valid_current_dates.index.max()
    current = sig_f[sig_f["date"] == current_date].sort_values("rank").copy()

    if len(bt_f) == 0:
        print("=" * 88)
        print("NO BACKTEST DATA IN SELECTED RANGE")
        print("=" * 88)
        print(f"Requested range: {start_date.date()} to {end_date.date()}")
        print(f"Adjusted range used: {adj_start_date.date()} to {adj_end_date.date()}")
        print(f"Loaded model data range: {loaded_min_date.date()} to {loaded_max_date.date()}")
        return

    perf = summarize_performance(bt_f)

    bt_f["strategy_cum"] = (1 + bt_f["strategy_ret"].fillna(0)).cumprod()
    bt_f["benchmark_cum"] = (1 + bt_f["benchmark_ret"].fillna(0)).cumprod()
    bt_f["spread_cum"] = (1 + bt_f["spread_ret"].fillna(0)).cumprod()

    print("=" * 88)
    print("SECTOR ROTATION DASHBOARD: MOMENTUM + ANALYST RECOMMENDATION MOMENTUM")
    print("=" * 88)
    print(f"Requested date range: {start_date.date()} to {end_date.date()}")
    print(f"Actual date range used: {adj_start_date.date()} to {adj_end_date.date()}")
    print(f"Loaded model data range: {loaded_min_date.date()} to {loaded_max_date.date()}")
    print(f"Lookback: {lookback_months} | Skip: {skip_months} | Top N: {top_n} | Bottom N: {bottom_n}")
    print(f"Factor weights: Momentum = {MOMENTUM_WEIGHT:.0%}, Analyst = {ANALYST_WEIGHT:.0%}")
    print(f"Latest valid signal date: {current_date.date()}")
    print("=" * 88)

    print("\nCURRENT MONTH SIGNAL TABLE")
    show_cols = [
        "date", "sector", "momentum_raw", "momentum_z",
        "analyst_raw", "analyst_z", "final_score",
        "rank", "signal", "n_stocks", "analyst_names", "data_source"
    ]
    current_show = current[show_cols].copy()
    for c in ["momentum_raw", "momentum_z", "analyst_raw", "analyst_z", "final_score"]:
        if c in current_show.columns:
            current_show[c] = current_show[c].round(4)
    display(current_show.reset_index(drop=True))

    print("\nBACKTEST SUMMARY")
    display_perf_table(perf)

    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["strategy_cum"], mode="lines", name="Strategy"))
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["benchmark_cum"], mode="lines", name="Benchmark"))
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["spread_cum"], mode="lines", name="Green-Red Spread"))
    fig1.update_layout(
        title="Cumulative Performance",
        xaxis_title="Date",
        yaxis_title="Growth of $1",
        template="plotly_white",
        width=1000,
        height=500
    )
    fig1.show()

    signal_to_num = {"Red": -1, "Yellow": 0, "Green": 1}
    hm = sig_f[["date", "sector", "signal"]].copy()
    hm["signal_num"] = hm["signal"].map(signal_to_num)
    heat = hm.pivot(index="sector", columns="date", values="signal_num").reindex(all_sectors)

    fig2 = px.imshow(
        heat,
        aspect="auto",
        title="Traffic Light Heatmap",
        color_continuous_scale=["red", "yellow", "green"],
        zmin=-1,
        zmax=1
    )
    fig2.update_layout(width=1200, height=500)
    fig2.show()

    fig3 = px.bar(
        current.sort_values("rank"),
        x="sector",
        y="final_score",
        color="signal",
        title=f"Current Combined Sector Score ({current_date.date()})",
        hover_data=["momentum_raw", "analyst_raw", "rank", "data_source"]
    )
    fig3.update_layout(template="plotly_white", width=1000, height=500)
    fig3.show()

    fig4 = px.bar(
        current.sort_values("rank"),
        x="sector",
        y=["momentum_z", "analyst_z"],
        barmode="group",
        title=f"Factor Decomposition ({current_date.date()})"
    )
    fig4.update_layout(template="plotly_white", width=1100, height=500)
    fig4.show()

    if SAVE_CSV_OUTPUTS:
        sig_f.to_csv("multifactor_sector_signals.csv", index=False)
        bt_f.to_csv("multifactor_sector_backtest.csv", index=False)
        current_show.to_csv("multifactor_current_signal_table.csv", index=False)
        print("\nCSV files saved to current Colab working directory.")

# -----------------------------
# 20) Widgets
# -----------------------------
available_dates = pd.Series(sorted(sector_factors["date"].dropna().unique()))
min_date = available_dates.min().date()
max_date = available_dates.max().date()

display(HTML("<h2>Sector Rotation Model Dashboard: Momentum + Analyst Recommendation Momentum</h2>"))

start_picker = widgets.DatePicker(description="Start", value=min_date)
end_picker = widgets.DatePicker(description="End", value=max_date)

lookback_slider = widgets.IntSlider(value=DEFAULT_LOOKBACK_MONTHS, min=3, max=12, step=1, description="Lookback")
skip_slider = widgets.IntSlider(value=DEFAULT_SKIP_MONTHS, min=0, max=2, step=1, description="Skip")
topn_slider = widgets.IntSlider(value=DEFAULT_TOP_N, min=1, max=5, step=1, description="Top N")
bottomn_slider = widgets.IntSlider(value=DEFAULT_BOTTOM_N, min=1, max=5, step=1, description="Bottom N")
run_button = widgets.Button(description="Run Dashboard", button_style="success")
output = widgets.Output()

def on_run_clicked(_):
    with output:
        render_dashboard(
            start_date=start_picker.value,
            end_date=end_picker.value,
            lookback_months=lookback_slider.value,
            skip_months=skip_slider.value,
            top_n=topn_slider.value,
            bottom_n=bottomn_slider.value
        )

run_button.on_click(on_run_clicked)

controls = widgets.VBox([
    widgets.HBox([start_picker, end_picker]),
    widgets.HBox([lookback_slider, skip_slider]),
    widgets.HBox([topn_slider, bottomn_slider]),
    widgets.HBox([run_button]),
])

display(controls, output)

with output:
    render_dashboard(
        start_date=start_picker.value,
        end_date=end_picker.value,
        lookback_months=lookback_slider.value,
        skip_months=skip_slider.value,
        top_n=topn_slider.value,
        bottom_n=bottomn_slider.value
    )

In [ ]:
# ============================================================
# TOP 3 SECTOR TIMELINE CHART
# Horizontal bars across time
# Color shows rank:
#   Top 1 = green
#   Top 2 = orange
#   Top 3 = red
# ============================================================

import pandas as pd
import plotly.express as px

def plot_top3_timeline(signal_df, start_date=None, end_date=None, top_n=3):
    sig = signal_df.copy()
    sig["date"] = pd.to_datetime(sig["date"]) + pd.offsets.MonthEnd(0)

    if start_date is not None:
        sig = sig[sig["date"] >= pd.to_datetime(start_date)]
    if end_date is not None:
        sig = sig[sig["date"] <= pd.to_datetime(end_date)]

    sig = sig.dropna(subset=["rank"]).copy()
    sig = sig[sig["rank"] <= top_n].copy()

    if sig.empty:
        print("No timeline data available in selected range.")
        return

    # One row per sector recommendation window
    rebalance_dates = sorted(sig["date"].unique())

    if len(rebalance_dates) < 2:
        print("Need at least two rebalance dates to build the timeline chart.")
        return

    rows = []
    for i in range(len(rebalance_dates) - 1):
        start_dt = pd.Timestamp(rebalance_dates[i])
        end_dt = pd.Timestamp(rebalance_dates[i + 1])

        month_picks = (
            sig[sig["date"] == start_dt]
            .sort_values("rank")
            .head(top_n)
            .copy()
        )

        for _, row in month_picks.iterrows():
            rank_val = int(row["rank"])
            rows.append({
                "sector": row["sector"],
                "start": start_dt,
                "end": end_dt,
                "rank_label": f"Top {rank_val}",
                "rank_num": rank_val,
                "window": start_dt.strftime("%Y-%m")
            })

    timeline_df = pd.DataFrame(rows)

    if timeline_df.empty:
        print("No timeline rows created.")
        return

    # Order sectors nicely
    sector_order = sorted(timeline_df["sector"].unique())

    color_map = {
        "Top 1": "green",
        "Top 2": "orange",
        "Top 3": "red"
    }

    fig = px.timeline(
        timeline_df,
        x_start="start",
        x_end="end",
        y="sector",
        color="rank_label",
        color_discrete_map=color_map,
        category_orders={"sector": sector_order, "rank_label": ["Top 1", "Top 2", "Top 3"]},
        hover_data=["window", "rank_label"]
    )

    fig.update_yaxes(autorange="reversed")

    fig.update_layout(
        title="Top 3 Sector Recommendations Across Time",
        xaxis_title="Date",
        yaxis_title="Sector",
        template="plotly_white",
        width=1400,
        height=650,
        legend_title="Recommendation Rank"
    )

    fig.show()


# ------------------------------------------------------------
# RUN
# ------------------------------------------------------------
PRESENTATION_START = "2024-01-01"
PRESENTATION_END   = "2026-4-30"
TOP_N_PRESENTATION = 3

plot_top3_timeline(
    signal_df=signals,
    start_date=PRESENTATION_START,
    end_date=PRESENTATION_END,
    top_n=TOP_N_PRESENTATION
)